<a href="https://colab.research.google.com/github/KijoSal-dev/Retrieval-Augmented-Generation-RAG-System-for-Question-Answering/blob/main/Salome_Kungu_CS_DA03_26054_WK12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install required libraries

In [1]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Upgrade and align the core LangChain ecosystem packages to resolve internal tracer mismatches
!pip install -q --upgrade "requests==2.32.4" langchain>=0.3.0 langchain-core>=0.3.0 langchain-community>=0.3.0 langchain-huggingface transformers sentence-transformers faiss-cpu pypdf

import re
import torch
from google.colab import files
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [2]:
# STEP 1: DEFINE CLEANING FUNCTION
# Fixes common PDF extraction errors like 'C y b e r' -> 'Cyber'

def fix_pdf_text(text):

    # Fix spacing issues from PDF extraction
    text = re.sub(r'(?<= )([^ ]) (?=[^ ])', r'\1', text)
    text = re.sub(r'^([^ ]) (?=[^ ])', r'\1', text)
    text = re.sub(r'\n([^ ]) ', r'\n\1', text)

    return re.sub(r'\s+', ' ', text).strip()

In [3]:
# STEP 2: Upload the PDF Document
print("Upload your PDF document")

uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]

print("Uploaded file:", pdf_path)

Upload your PDF document


Saving National-Agriculture-Production-Report-2025.pdf to National-Agriculture-Production-Report-2025 (2).pdf
Uploaded file: National-Agriculture-Production-Report-2025 (2).pdf


In [4]:
# STEP 3: Load and Process the PDF
loader = PyPDFLoader(pdf_path)

pages = loader.load()

print("Number of pages loaded:", len(pages))

# Clean the text
for page in pages:
    page.page_content = fix_pdf_text(page.page_content)

Number of pages loaded: 202


In [5]:
# STEP 4: Split the Document into Chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100
)

chunks = splitter.split_documents(pages)

print("Number of chunks created:", len(chunks))

Number of chunks created: 558


In [6]:
# STEP 5: Create Embeddings and FAISS Vector Store
print("Creating embeddings and vector store...")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k":3}
)

print("Vector store created successfully")

Creating embeddings and vector store...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store created successfully


In [7]:
# STEP 6: Load the Generative AI Model (FLAN-T5)
device = "cuda" if torch.cuda.is_available() else "cpu"

model_id = "google/flan-t5-large"

print("Loading model:", model_id)

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device)

print("Model loaded on:", device)

Loading model: google/flan-t5-large


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model loaded on: cpu


In [8]:
# STEP 7: Create the RAG Query Function
def query_rag(question):

    # Retrieve relevant chunks
    relevant_docs = retriever.invoke(question)

    # Combine retrieved text
    context = "\n".join([doc.page_content for doc in relevant_docs])

    # Prompt for the LLM
    input_text = f"answer the question based on the context below.\n\nQuestion: {question}\n\nContext: {context}"

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    # Generate response
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer

In [14]:
def query_rag_v2(question, temperature=0.0, verbose=True):
    # 1. Retrieve the top relevant chunks
    relevant_docs = retriever.invoke(question)

    # Extract source page numbers for simple logging
    sources = [doc.metadata.get("page", "Unknown") for doc in relevant_docs]

    # Combine retrieved text
    context = "\n".join([doc.page_content for doc in relevant_docs])

    # 2. Structured Prompt with simple guardrails to prevent hallucinations
    input_text = (
        "You are an honest assistant. Answer the question using ONLY the provided context.\n"
        "Rules:\n"
        "1. If the context does not contain the answer, reply exactly with: 'I cannot find the answer in the provided report.'\n"
        "2. Do not make up information.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    # 3. Flexible generation settings
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=temperature > 0.0, # Enable sampling if temperature > 0
        temperature=temperature if temperature > 0.0 else None
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Simple beginner-friendly logger
    if verbose:
        print("\n--- LOGGING INFO ---")
        print(f"Retrieved from Pages: {sources}")
        print(f"Snippet matched: '{relevant_docs[0].page_content[:150]}...'\n")

    return answer

# Quick Demo of the new simplified features!
print("--- Demo with an Answerable Question ---")
print(query_rag_v2("How much did the agricultural sector contribute to Kenya's GDP in 2024?", verbose=True))

print("\n--- Demo with an Unanswerable Question (Testing Guardrails) ---")
print(query_rag_v2("How many tractors were imported from Brazil in 2023?", verbose=False))

--- Demo with an Answerable Question ---

--- LOGGING INFO ---
Retrieved from Pages: [24, 13, 24]
Snippet matched: 'National Agriculture Production Report 2025 1 Overview The agricultural sector remains the main sector sup - porting Kenya’s economy, contributing 22....'

22.5 per cent

--- Demo with an Unanswerable Question (Testing Guardrails) ---
I cannot find the answer in the provided report.


In [9]:
# Test set
test_set = [
    # --- SECTION 1: Food Crops & Staples ---
    {
        "id": 1,
        "question": "What was the total area under maize production and its total output in 2024?",
        "ground_truth": "Retrieved from Table 3.2 (Area, Production and Value of Maize)",
        "target_chunk_id": 11,  # Adjust based on your PDF chunk index
        "is_unanswerable": False,
    },
    {
        "id": 2,
        "question": "How much did the agricultural sector contribute to Kenya's GDP in 2024 according to the report?",
        "ground_truth": "22.5 per cent",
        "target_chunk_id": 3,
        "is_unanswerable": False,
    },
    {
        "id": 3,
        "question": "What were the trends in production volume and value for beans from 2020 to 2024?",
        "ground_truth": "Retrieved from Table 3.10 (Beans Production and Value)",
        "target_chunk_id": 18,
        "is_unanswerable": False,
    },
    {
        "id": 4,
        "question": "What are the core production figures for rice and the performance of the Mwea Irrigation Scheme?",
        "ground_truth": "Retrieved from Tables 3.4 and 3.5 (Rice and Mwea Scheme)",
        "target_chunk_id": 12,
        "is_unanswerable": False,
    },
    {
        "id": 5,
        "question": "What was the total production volume for sorghum and millet in 2024?",
        "ground_truth": "Retrieved from Tables 3.7 and 3.8 (Millet and Sorghum Production)",
        "target_chunk_id": 14,
        "is_unanswerable": False,
    },
    {
        "id": 6,
        "question": "How did green grams and cowpeas production shift across the 2020-2024 period?",
        "ground_truth": "Retrieved from Tables 3.11 and 3.12",
        "target_chunk_id": 21,
        "is_unanswerable": False,
    },

    # --- SECTION 2: Horticulture & Export Crops ---
    {
        "id": 7,
        "question": "What are the leading categories and total volumes of horticulture exports by destination in 2024?",
        "ground_truth": "Retrieved from Tables 4.10 and 4.11 (Horticulture Exports)",
        "target_chunk_id": 53,
        "is_unanswerable": False,
    },
    {
        "id": 8,
        "question": "What is the breakdown of production areas and values for Exotic versus African Leafy Vegetables (ALVs)?",
        "ground_truth": "Retrieved from Tables 4.5 and 4.6",
        "target_chunk_id": 40,
        "is_unanswerable": False,
    },

    # --- SECTION 3: Industrial Crops ---
    {
        "id": 9,
        "question": "What was the total area under coffee and summary of direct sales versus auction volumes up to 2023/24?",
        "ground_truth": "Retrieved from Tables 5.1, 5.1.2, and 5.1.3",
        "target_chunk_id": 57,
        "is_unanswerable": False,
    },
    {
        "id": 10,
        "question": "What were the smallholder and estate production quantities for made tea and green leaf in 2024?",
        "ground_truth": "Retrieved from Table 5.2.1(a)",
        "target_chunk_id": 63,
        "is_unanswerable": False,
    },
    {
        "id": 11,
        "question": "What were the cane deliveries by factory and total sugar production by county in 2024?",
        "ground_truth": "Retrieved from Tables 5.3.1 and 5.3.3",
        "target_chunk_id": 72,
        "is_unanswerable": False,
    },
    {
        "id": 12,
        "question": "What are the tracking figures for sisal production summary and quantities by county?",
        "ground_truth": "Retrieved from Tables 5.4(a) and 5.4(b)",
        "target_chunk_id": 78,
        "is_unanswerable": False,
    },
    {
        "id": 13,
        "question": "What are the seed cotton production yields and lint values recorded between 2020 and 2024?",
        "ground_truth": "Retrieved from Tables 5.5(a) and 5.5(b)",
        "target_chunk_id": 85,
        "is_unanswerable": False,
    },

    # --- SECTION 4: Nuts, Oil Crops, Livestock & Fisheries ---
    {
        "id": 14,
        "question": "What are the area, production, and value numbers for Macadamia and Cashew nuts by county?",
        "ground_truth": "Retrieved from Tables 6.2 and 6.3",
        "target_chunk_id": 104,
        "is_unanswerable": False,
    },
    {
        "id": 15,
        "question": "What was the total milk production and milk intake recorded by processors from 2020 to 2024?",
        "ground_truth": "Retrieved from Table 7.9 (Milk Production and Intake)",
        "target_chunk_id": 131,
        "is_unanswerable": False,
    },
    {
        "id": 16,
        "question": "What are the total quantities and estimated values of meat produced from livestock slaughter in 2024?",
        "ground_truth": "Retrieved from Table 7.11 (Meat Produced and Value)",
        "target_chunk_id": 134,
        "is_unanswerable": False,
    },
    {
        "id": 17,
        "question": "What are the production dynamics and values of fresh fish species across major water bodies up to 2024?",
        "ground_truth": "Retrieved from Tables 8.2 and 8.3",
        "target_chunk_id": 152,
        "is_unanswerable": False,
    },

    # --- SECTION 5: Unanswerable Questions (Hallucination & Honesty Check) ---
    {
        "id": 18,
        "question": "What was the total government expenditure in billion KSh allocated specifically to greenhouse subsidies in Nakuru County for 2024?",
        "ground_truth": "Unanswerable / Not in report",
        "target_chunk_id": None,
        "is_unanswerable": True,
    },
    {
        "id": 19,
        "question": "How many tractors were imported directly from Brazil by large-scale wheat farms in Narok during 2023?",
        "ground_truth": "Unanswerable / Not in report",
        "target_chunk_id": None,
        "is_unanswerable": True,
    },
    {
        "id": 20,
        "question": "What was the precise market share percentage held by private agricultural tech startups in the agritech sector in 2024?",
        "ground_truth": "Unanswerable / Not in report",
        "target_chunk_id": None,
        "is_unanswerable": True,
    },
]

In [16]:
import time
import pandas as pd

evaluation_results_v2 = []
k = 3  # Top-k retrieved chunks

print("Evaluating test set with query_rag_v2 (please wait as it runs on CPU)...\n")

for item in test_set:
    question = item["question"]

    # 1. Measure Retrieval Latency & Results
    start_retrieval = time.time()
    retrieved_docs = retriever.invoke(question)
    retrieval_time = time.time() - start_retrieval

    # Extract page index from document metadata
    retrieved_chunk_ids = [doc.metadata.get("page") for doc in retrieved_docs]

    # Compute Hit Rate @ k (1 if target is in top k, else 0)
    hit_at_k = 1 if item["target_chunk_id"] in retrieved_chunk_ids else 0

    # Compute MRR (Mean Reciprocal Rank)
    mrr = 0.0
    if item["target_chunk_id"] in retrieved_chunk_ids:
        rank = retrieved_chunk_ids.index(item["target_chunk_id"]) + 1
        mrr = 1.0 / rank

    # 2. Measure Generation Latency & Response
    start_gen = time.time()
    # Using our updated query function with safety guardrails!
    response = query_rag_v2(question, temperature=0.0, verbose=False)
    generation_time = time.time() - start_gen

    evaluation_results_v2.append({
        "id": item["id"],
        "question": question,
        "ground_truth": item["ground_truth"],
        "is_unanswerable": item["is_unanswerable"],
        "hit_at_k": hit_at_k,
        "mrr": mrr,
        "retrieval_time": retrieval_time,
        "generation_time": generation_time,
        "retrieved_chunks": retrieved_chunk_ids,
        "generated_answer": response,
    })

    print(f"Processed Q{item['id']}: {question[:50]}... -> Done")

# Compute and display overall summary metrics
avg_hit_rate = (sum([res["hit_at_k"] for res in evaluation_results_v2]) / len(evaluation_results_v2)) * 100
avg_mrr = sum([res["mrr"] for res in evaluation_results_v2]) / len(evaluation_results_v2)
avg_latency = sum([res["retrieval_time"] + res["generation_time"] for res in evaluation_results_v2]) / len(evaluation_results_v2)

print("\n=== FINAL EVALUATION SUMMARY ===")
print(f"Hit Rate @ {k}: {avg_hit_rate:.1f}%")
print(f"Mean Reciprocal Rank (MRR): {avg_mrr:.2f}")
print(f"Average Total Latency: {avg_latency:.2f} seconds")

# Save to a fresh CSV file
df_v2 = pd.DataFrame(evaluation_results_v2)
df_v2.to_csv("rag_evaluation_v2.csv", index=False)
print("\nSaved evaluation results to: rag_evaluation_v2.csv")

Evaluating test set with query_rag_v2 (please wait as it runs on CPU)...

Processed Q1: What was the total area under maize production and... -> Done
Processed Q2: How much did the agricultural sector contribute to... -> Done
Processed Q3: What were the trends in production volume and valu... -> Done
Processed Q4: What are the core production figures for rice and ... -> Done
Processed Q5: What was the total production volume for sorghum a... -> Done
Processed Q6: How did green grams and cowpeas production shift a... -> Done
Processed Q7: What are the leading categories and total volumes ... -> Done
Processed Q8: What is the breakdown of production areas and valu... -> Done
Processed Q9: What was the total area under coffee and summary o... -> Done
Processed Q10: What were the smallholder and estate production qu... -> Done
Processed Q11: What were the cane deliveries by factory and total... -> Done
Processed Q12: What are the tracking figures for sisal production... -> Done
Processed Q

### Beginner-Friendly Chunk Size Experiment
In this experiment, we will split our documents into three different chunk sizes: 300, 700, and 1000 characters. For each size, we will measure how long it takes to retrieve the context and evaluate whether it successfully fetches the correct information.

In [17]:
import time

# Let's select a few sample questions to test
sample_questions = [
    {
        "question": "How much did the agricultural sector contribute to Kenya's GDP in 2024 according to the report?",
        "expected_page_contains": "22.5"
    },
    {
        "question": "What are the core production figures for rice and the performance of the Mwea Irrigation Scheme?",
        "expected_page_contains": "Mwea"
    }
]

chunk_sizes = [300, 700, 1000]
experiment_results = []

for size in chunk_sizes:
    print(f"--- Running Experiment for Chunk Size: {size} ---")

    # 1. Re-split pages
    temp_splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=int(size * 0.15)
    )
    temp_chunks = temp_splitter.split_documents(pages)

    # 2. Re-index with FAISS
    temp_vectorstore = FAISS.from_documents(temp_chunks, embeddings)
    temp_retriever = temp_vectorstore.as_retriever(search_kwargs={"k": 3})

    # 3. Test Retrieval
    for idx, sample in enumerate(sample_questions):
        start_time = time.time()
        docs = temp_retriever.invoke(sample["question"])
        ret_time = time.time() - start_time

        # Check if expected word is found in the retrieved context
        full_text = " ".join([d.page_content for d in docs])
        is_success = sample["expected_page_contains"].lower() in full_text.lower()

        experiment_results.append({
            "chunk_size": size,
            "question_id": idx + 1,
            "retrieval_time_seconds": ret_time,
            "successfully_retrieved_target_info": is_success
        })
        print(f"  Q{idx+1}: Time={ret_time:.4f}s | Success={is_success}")

# Display the structured results comparison
df_exp = pd.DataFrame(experiment_results)
display(df_exp)

--- Running Experiment for Chunk Size: 300 ---
  Q1: Time=0.0285s | Success=True
  Q2: Time=0.0270s | Success=True
--- Running Experiment for Chunk Size: 700 ---
  Q1: Time=0.0249s | Success=True
  Q2: Time=0.0236s | Success=True
--- Running Experiment for Chunk Size: 1000 ---
  Q1: Time=0.0334s | Success=True
  Q2: Time=0.0322s | Success=True


,chunk_size,question_id,retrieval_time_seconds,successfully_retrieved_target_info
0,300,1,0.028548,True
1,300,2,0.026955,True
2,700,1,0.024869,True
3,700,2,0.023553,True
4,1000,1,0.033424,True
5,1000,2,0.032157,True


In [10]:
# Evaluation Loop for Hit Rate, MRR, and Latency

import time

evaluation_results = []
k = 3  # Top-k retrieved chunks

for item in test_set:
  question = item["question"]

  # 1. Measure Retrieval Latency & Results
  start_retrieval = time.time()
  # Use the active retriever from your notebook
  retrieved_docs = retriever.invoke(question)
  retrieval_time = time.time() - start_retrieval

  # Retrieve the page/chunk indexes from metadata
  retrieved_chunk_ids = [doc.metadata.get("page") for doc in retrieved_docs]

  # Compute Hit Rate @ k (1 if target is in top k, else 0)
  hit_at_k = 1 if item["target_chunk_id"] in retrieved_chunk_ids else 0

  # Compute MRR (Mean Reciprocal Rank)
  mrr = 0.0
  if item["target_chunk_id"] in retrieved_chunk_ids:
    rank = retrieved_chunk_ids.index(item["target_chunk_id"]) + 1
    mrr = 1.0 / rank

  # 2. Measure Generation Latency & Response
  start_gen = time.time()
  # Use your active query_rag function
  response = query_rag(question)
  generation_time = time.time() - start_gen

  evaluation_results.append({
      "question": question,
      "hit_at_k": hit_at_k,
      "mrr": mrr,
      "retrieval_time": retrieval_time,
      "generation_time": generation_time,
      "retrieved_chunks": retrieved_chunk_ids,
      "generated_answer": response,
  })

# Calculate final averages
avg_hit_rate = (
    sum([res["hit_at_k"] for res in evaluation_results]) / len(evaluation_results)
) * 100
avg_mrr = sum([res["mrr"] for res in evaluation_results]) / len(
    evaluation_results
)
avg_latency = sum(
    [res["retrieval_time"] + res["generation_time"] for res in evaluation_results]
) / len(evaluation_results)

print(f"Hit Rate @ {k}: {avg_hit_rate:.1f}%")
print(f"Mean Reciprocal Rank (MRR): {avg_mrr:.2f}")
print(f"Average Latency: {avg_latency:.2f} seconds")

Hit Rate @ 3: 5.0%
Mean Reciprocal Rank (MRR): 0.05
Average Latency: 18.91 seconds


In [13]:
# Manual Grading Checklist

import pandas as pd

df = pd.DataFrame(evaluation_results)
df.to_csv("rag_evaluation_template.csv", index=False)
print("Saved evaluation template. Open CSV to fill manual scores.")

Saved evaluation template. Open CSV to fill manual scores.
